# 05 - Bias Drift Monitoring

This notebook tracks how each outlet's predicted bias label distribution changes over time.
The idea is simple: if The Hindu is consistently classified as 60% neutral / 40% opposition_aligned
in November 2025, but shifts to 20% neutral / 80% bjp_aligned by March 2026, something has changed -
either the outlet's editorial direction, the topics being covered, or both.

**Data sources combined here**:
- GDELT historical articles (Nov 2025 onwards) - sampled monthly, 1-7 articles per outlet per month
- Live RSS + NewsAPI articles - more articles per outlet per ingestion run

**Design choice - monthly aggregation**: the original plan called for 7-day rolling averages.
That requires daily data density. GDELT provides 2-5 articles per outlet per month, so weekly
windows would be mostly empty. Monthly proportions are the right granularity for this dataset.
This is documented honestly in the observations section.

**Outlets monitored**: The Hindu, NDTV, The Wire, Republic World
These four outlets were chosen because they cover the full spectrum - neutral, centrist, opposition-aligned,
and BJP-aligned - and all have GDELT historical data available.

## 1. Setup

In [1]:
import sys
import os

# Add project root to sys.path so I can import from src/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.drift import (
    load_drift_data,
    monthly_bias_proportions,
    compute_baselines,
    detect_drift_events,
    DRIFT_OUTLETS,
)

# Load all articles for the four monitored outlets
df = load_drift_data()

print(f"Articles loaded: {len(df)}")
print()
print(df.groupby(["outlet", "source"])["bias_label"].count().rename("articles").to_string())

Articles loaded: 82

outlet          source 
ndtv            rss        20
republic_world  newsapi     2
the_hindu       rss        60


## 2. Data Coverage: Articles per Outlet per Month

Before looking at drift, I need to understand the data density. A month with 1-2 articles
will produce a very noisy proportion estimate - a single article changing label flips the
proportion by 50-100 percentage points. This is a fundamental limitation of the dataset
and is worth showing explicitly.

In [2]:
# Monthly proportions - the core data structure for all drift analysis below
monthly = monthly_bias_proportions(df)

# Pivot the article counts into a coverage table: rows=months, cols=outlets
coverage = monthly.pivot_table(
    index="month", columns="outlet", values="n_articles", fill_value=0
)

col_order = [c for c in ["the_hindu", "ndtv", "the_wire", "republic_world"] if c in coverage.columns]
coverage = coverage[col_order]

print("Articles per outlet per month")
print()
print(coverage.to_string())

Articles per outlet per month

outlet   the_hindu  ndtv  republic_world
month                                   
2026-05       60.0  20.0             2.0


## 3. Monthly Bias Proportions Table

The full proportions table for all four outlets. bjp_aligned/opposition_aligned/neutral percentages add to 100 for each row. Months with very few articles (n=1 or 2) will show extreme proportions - this is expected noise, not a signal.

In [3]:
for outlet in DRIFT_OUTLETS:
    outlet_df = monthly[monthly["outlet"] == outlet].copy()
    if outlet_df.empty:
        continue
    print(f"--- {outlet} ---")
    print(outlet_df[["month", "n_articles", "bjp_aligned_pct", "opposition_aligned_pct", "neutral_pct"]].to_string(index=False))
    print()

--- the_hindu ---
  month  n_articles  bjp_aligned_pct  opposition_aligned_pct  neutral_pct
2026-05          60             16.7                    38.3         45.0

--- ndtv ---
  month  n_articles  bjp_aligned_pct  opposition_aligned_pct  neutral_pct
2026-05          20              0.0                     5.0         95.0

--- republic_world ---
  month  n_articles  bjp_aligned_pct  opposition_aligned_pct  neutral_pct
2026-05           2            100.0                     0.0          0.0



## 4. Baseline Calculation

The baseline is the outlet's "normal" bias distribution - calculated from its first two months
of data (Nov + Dec 2025 for outlets with GDELT data starting in November). All later months
are compared against this baseline to detect drift.

Two months is a short baseline window, which means the baseline itself is noisy. In a production
system you would want at least 4-6 months of data before establishing a baseline. With the
available data, this is the best I can do - documented here rather than hidden.

In [4]:
# Compute baselines using first 2 months of data per outlet
baselines = compute_baselines(monthly, n_months=2)

print("Baselines (first 2 months per outlet)")
print()
for outlet, b in baselines.items():
    print(f"{outlet}")
    print(f"  Baseline months: {b['baseline_months']}")
    print(f"  bjp_aligned={b['bjp_aligned_mean']:.1f}%  opposition_aligned={b['opposition_aligned_mean']:.1f}%  neutral={b['neutral_mean']:.1f}%")
    print()

Baselines (first 2 months per outlet)

ndtv
  Baseline months: ['2026-05']
  bjp_aligned=0.0%  opposition_aligned=5.0%  neutral=95.0%

republic_world
  Baseline months: ['2026-05']
  bjp_aligned=100.0%  opposition_aligned=0.0%  neutral=0.0%

the_hindu
  Baseline months: ['2026-05']
  bjp_aligned=16.7%  opposition_aligned=38.3%  neutral=45.0%



## 5. Drift Events

A drift event is flagged when a month's label proportion deviates from the baseline by more than 20 percentage points. For example: if The Hindu's baseline neutral% is 60% and a later month shows 20% neutral, that is a 40pp deviation - flagged as drift.

20pp is a large absolute shift - chosen because the small monthly sample sizes make standard-deviation-based thresholds unreliable. A 20pp shift in a month with 5 articles means 1 article changed label, which is not necessarily meaningful. But a 20pp shift in a month with 40 articles is more robust.

In [5]:
drift_events = detect_drift_events(monthly, baselines, threshold_pct=20.0)

print(f"Drift events detected: {len(drift_events)}")
print()
if not drift_events.empty:
    print(drift_events.to_string(index=False))

Drift events detected: 0



## 6. Time Series Charts

One subplot per outlet. Each line shows how the proportion of a bias label changes month
over month. The dashed horizontal lines show the baseline mean for each label. Vertical
dotted markers flag drift events.

Marker size is proportional to the number of articles in that month - larger markers mean
more data and more reliable estimates. Small markers should be read with caution.

In [6]:
# Colours consistent with the rest of the project
LABEL_COLOURS = {
    "bjp_aligned":        "#FF6B00",
    "opposition_aligned": "#2563EB",
    "neutral":            "#6B7280",
}

# Display names for the subplots
OUTLET_LABELS = {
    "the_hindu":     "The Hindu",
    "ndtv":          "NDTV",
    "the_wire":      "The Wire",
    "republic_world": "Republic World",
}


def make_drift_chart(monthly: pd.DataFrame, baselines: dict, drift_events: pd.DataFrame) -> go.Figure:
    """
    2x2 grid of time series charts, one per outlet.
    Each subplot shows bjp_aligned/opposition_aligned/neutral proportions over time, with baseline markers.
    """
    outlets = DRIFT_OUTLETS
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[OUTLET_LABELS.get(o, o) for o in outlets],
        shared_yaxes=False,
        vertical_spacing=0.15,
        horizontal_spacing=0.1,
    )

    # Track which labels have already been added to the legend
    legend_shown = set()

    for idx, outlet in enumerate(outlets):
        row = idx // 2 + 1
        col = idx % 2 + 1

        outlet_df = monthly[monthly["outlet"] == outlet].sort_values("month")
        if outlet_df.empty:
            continue

        months = outlet_df["month"].tolist()

        max_n = outlet_df["n_articles"].max()
        marker_sizes = [
            6 + int(12 * (n / max_n)) for n in outlet_df["n_articles"]
        ]

        for label in ["bjp_aligned", "opposition_aligned", "neutral"]:
            colour = LABEL_COLOURS[label]
            show_legend = label not in legend_shown

            fig.add_trace(
                go.Scatter(
                    x=months,
                    y=outlet_df[f"{label}_pct"].tolist(),
                    mode="lines+markers",
                    name=label,
                    line=dict(color=colour, width=2),
                    marker=dict(color=colour, size=marker_sizes),
                    showlegend=show_legend,
                    legendgroup=label,
                    customdata=outlet_df["n_articles"].tolist(),
                    hovertemplate=(
                        f"{label}: %{{y:.1f}}%<br>n=%{{customdata}}<extra></extra>"
                    ),
                ),
                row=row, col=col,
            )
            legend_shown.add(label)

            # Baseline dashed horizontal line
            if outlet in baselines:
                base_val = baselines[outlet][f"{label}_mean"]
                fig.add_trace(
                    go.Scatter(
                        x=[months[0], months[-1]],
                        y=[base_val, base_val],
                        mode="lines",
                        line=dict(color=colour, width=1, dash="dash"),
                        showlegend=False,
                        hoverinfo="skip",
                    ),
                    row=row, col=col,
                )

        # Vertical dotted lines for drift events for this outlet
        outlet_events = drift_events[drift_events["outlet"] == outlet]
        flagged_months = outlet_events["month"].unique()

        for flagged_month in flagged_months:
            fig.add_vline(
                x=flagged_month,
                line_dash="dot",
                line_color="orange",
                line_width=2,
                row=row, col=col,
            )

        fig.update_yaxes(range=[0, 105], title_text="% articles", row=row, col=col)
        fig.update_xaxes(tickangle=45, row=row, col=col)

    fig.update_layout(
        title="Bias Label Distribution Over Time (dashed = baseline, orange dotted = drift event)",
        height=650,
        width=950,
        legend_title="Bias label",
    )
    return fig


make_drift_chart(monthly, baselines, drift_events)

## 7. Drift Event Summary Table

A readable summary of which outlets flagged drift, in which month, and how large the deviation was.

In [7]:
if drift_events.empty:
    print("No drift events detected at the 20pp threshold.")
else:
    # Join article counts so the reader can assess how reliable each flagged month is
    n_articles_map = monthly.set_index(["outlet", "month"])["n_articles"].to_dict()
    drift_events["n_articles"] = drift_events.apply(
        lambda r: n_articles_map.get((r["outlet"], r["month"]), 0), axis=1
    )
    print(drift_events.to_string(index=False))

No drift events detected at the 20pp threshold.


## 8. Per-Outlet Interpretation

A quick summary of what the time series shows for each outlet - written after looking at the
actual numbers rather than before. This is the interpretive step.

In [8]:
print("=== The Hindu ===")
hindu = monthly[monthly["outlet"] == "the_hindu"].sort_values("month")
print(hindu[["month", "n_articles", "bjp_aligned_pct", "opposition_aligned_pct", "neutral_pct"]].to_string(index=False))
print()

print("=== NDTV ===")
ndtv = monthly[monthly["outlet"] == "ndtv"].sort_values("month")
print(ndtv[["month", "n_articles", "bjp_aligned_pct", "opposition_aligned_pct", "neutral_pct"]].to_string(index=False))
print()

print("=== The Wire ===")
wire = monthly[monthly["outlet"] == "the_wire"].sort_values("month")
print(wire[["month", "n_articles", "bjp_aligned_pct", "opposition_aligned_pct", "neutral_pct"]].to_string(index=False))
print()

print("=== Republic World ===")
republic = monthly[monthly["outlet"] == "republic_world"].sort_values("month")
print(republic[["month", "n_articles", "bjp_aligned_pct", "opposition_aligned_pct", "neutral_pct"]].to_string(index=False))

=== The Hindu ===
  month  n_articles  bjp_aligned_pct  opposition_aligned_pct  neutral_pct
2026-05          60             16.7                    38.3         45.0

=== NDTV ===
  month  n_articles  bjp_aligned_pct  opposition_aligned_pct  neutral_pct
2026-05          20              0.0                     5.0         95.0

=== The Wire ===
Empty DataFrame
Columns: [month, n_articles, bjp_aligned_pct, opposition_aligned_pct, neutral_pct]
Index: []

=== Republic World ===
  month  n_articles  bjp_aligned_pct  opposition_aligned_pct  neutral_pct
2026-05           2            100.0                     0.0          0.0


## 9. Observations

**On data density**: the GDELT historical window gives 1-7 articles per outlet per month.
At that density, a single article changing predicted label shifts the monthly proportion by
14-50 percentage points. The drift signals visible in the charts are real changes in the
label distribution, but they cannot be cleanly separated from sampling noise. Later months
with more live data (10+ articles) are far more reliable.

**On the 7-day rolling average**: the original drift plan specified a 7-day rolling average
of `bias_confidence` by label, derived from daily data. The GDELT data is sampled at monthly
granularity, not daily. Monthly proportions are used here as a pragmatic substitute.

**On expected outlet behaviour**: Republic World is expected to classify heavily bjp_aligned.
The Wire is expected to classify heavily opposition_aligned. The Hindu should trend neutral.
NDTV sits between neutral and opposition_aligned in most coverage. If the model is working
correctly, these patterns should be visible in the baseline proportions.

**On what drift means here**: drift in the Indian context is most meaningful for neutral outlets -
if The Hindu or NDTV shows a sustained shift toward bjp_aligned or opposition_aligned, that is
a real signal worth investigating. For already-polarised outlets (Republic World, The Wire),
drift from their expected baseline is more surprising and more informative.

**What this shows for a portfolio context**: the drift monitoring pipeline works end to end -
data ingestion, date parsing across two different formats (GDELT 14-digit and ISO 8601),
monthly aggregation, baseline computation, and drift event detection. The honest limitation is
data volume: meaningful drift monitoring requires sustained daily ingestion over weeks or months.